In [ ]:
import pandas as pd
import os

# 📂 폴더 경로
folder_path = r"C:\Users\manid\OneDrive\바탕 화면\data_study\kdt"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# 결과 리스트
result_rows = []

for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)
    
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp949')
    df = df.fillna('')

    # 내부직원 필터링
    is_test_reason = df['지원취소 사유'].str.lower().str.contains('test|테스트', case=False)
    is_internal_email = df['지원서 이메일'].str.contains('likelion.net', case=False) | df['가입 이메일'].str.contains('likelion.net', case=False)
    is_internal = is_test_reason | is_internal_email

    # 지표 계산
    cond_completed = (df['지원완료일'] != '') & (df['지원상태'] != '지원취소') & (~is_internal)
    cond_applying = df['합불상태'].isin(['검토전', '대상아님']) & (df['지원완료일'] == '') & (~is_internal)
    cond_canceled_valid = (df['지원상태'] == '지원취소') & (df['지원완료일'] != '') & (~is_internal)


    # ✅ 내배카 합계 조건 수정: yes/no + 내부직원 제외 + 지원중 or 지원완료만 포함
    cond_nbc_yes_no = (
        df['내배카 보유'].str.lower().isin(['yes', 'no']) &
        (~is_internal) &
        (df['지원상태'].isin(['지원완료', '지원중']))
    )

    지원완료수 = df[cond_completed].shape[0]
    지원중수 = df[cond_applying].shape[0]
    유효_지원취소수 = df[cond_canceled_valid].shape[0]
    지원시작수 = 지원완료수 + 지원중수 + 유효_지원취소수
    내배카_합계 = df[cond_nbc_yes_no].shape[0]
    부트캠프명 = os.path.splitext(file_name)[0]

    result_rows.append({
        '부트캠프명': 부트캠프명,
        '지원시작': 지원시작수,
        '지원중': 지원중수,
        '내배카 합계': 내배카_합계,
        '지원완료': 지원완료수
    })

# 결과 표 생성
summary_df = pd.DataFrame(result_rows)

# 보기 좋게 정렬
summary_df = summary_df.sort_values(by='부트캠프명').reset_index(drop=True)

# 📊 주피터에서 테이블 출력
display(summary_df)

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# 🔠 한글 폰트 설정
if platform.system() == 'Windows':
    font_path = "C:/Windows/Fonts/malgun.ttf"
elif platform.system() == 'Darwin':
    font_path = "/System/Library/Fonts/AppleGothic.ttf"
else:
    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

fontprop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = fontprop.get_name()
plt.rcParams['axes.unicode_minus'] = False

# 📂 폴더 경로
folder_path = r"C:\Users\manid\OneDrive\바탕 화면\data_study\kdt"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

plt.style.use('ggplot')

# 🔁 파일 반복
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)

    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp949')
    df = df.fillna('')

    # 날짜 변환
    df['최초작성일'] = pd.to_datetime(df['최초작성일'], errors='coerce')
    df['지원완료일'] = pd.to_datetime(df['지원완료일'], errors='coerce')

    # 내부직원 필터링
    is_test_reason = df['지원취소 사유'].str.lower().str.contains('test|테스트', case=False)
    is_internal_email = df['지원서 이메일'].str.contains('likelion.net', case=False) | df['가입 이메일'].str.contains('likelion.net', case=False)
    is_internal = is_test_reason | is_internal_email

    # 조건 정의
    cond_applying = df['합불상태'].isin(['검토전', '대상아님']) & (~is_internal)
    cond_completed_start = (df['지원상태'] == '지원완료') & (df['지원완료일'].notna()) & (~is_internal)
    cond_canceled_valid = (df['지원상태'] == '지원취소') & (df['지원완료일'].notna()) & (~is_internal)
    cond_completed = (df['지원완료일'].notna()) & (df['지원상태'] != '지원취소') & (~is_internal)

    # ▶ 지원시작수: 최초작성일 기준
    df_started = df[cond_applying | cond_completed_start | cond_canceled_valid].copy()
    started_by_day = df_started.groupby(df_started['최초작성일'].dt.date).size()

    # ▶ 지원완료수: 지원완료일 기준
    df_completed = df[cond_completed].copy()
    completed_by_day = df_completed.groupby(df_completed['지원완료일'].dt.date).size()

    # ▶ 내일배움카드 YES 비율
    cond_nbc_yes = (
        df['내배카 보유'].str.lower() == 'yes'
    ) & (~is_internal) & (df['지원상태'].isin(['지원완료', '지원중']))

    df_nbc_yes = df[cond_nbc_yes].copy()
    nbc_yes_by_day = df_nbc_yes.groupby(df_nbc_yes['최초작성일'].dt.date).size()

    # ▶ 통합 데이터프레임 구성
    daily_df = pd.DataFrame({
        '지원시작수': started_by_day,
        '지원완료수': completed_by_day,
        '내배카 YES': nbc_yes_by_day
    }).fillna(0).astype(int)

    # ▶ 비율(%) 계산
    daily_df['내배카 YES 비율(%)'] = (daily_df['내배카 YES'] / daily_df['지원시작수']) * 100
    daily_df['누적 지원시작수'] = daily_df['지원시작수'].cumsum()
    daily_df['누적 지원완료수'] = daily_df['지원완료수'].cumsum()

    # ▶ 시각화
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # ▶ 왼쪽 축: 누적 지원자 수
    ax1.plot(daily_df.index, daily_df['누적 지원시작수'], label='누적 지원시작수', color='tomato', marker='o')
    ax1.plot(daily_df.index, daily_df['누적 지원완료수'], label='누적 지원완료수', color='steelblue', marker='s')

    for i, value in enumerate(daily_df['누적 지원시작수']):
        ax1.text(daily_df.index[i], value + 2, str(value), ha='center', fontsize=8, color='tomato')
    for i, value in enumerate(daily_df['누적 지원완료수']):
        ax1.text(daily_df.index[i], value + 2, str(value), ha='center', fontsize=8, color='steelblue')

    ax1.set_xlabel("날짜")
    ax1.set_ylabel("누적 지원자 수")
    ax1.tick_params(axis='x', rotation=45)
    ax1.legend(loc='upper left')
    ax1.grid(True)

    # ▶ 오른쪽 축: 일일 내배카 YES 비율
    ax2 = ax1.twinx()
    ax2.plot(daily_df.index, daily_df['내배카 YES 비율(%)'], label='내배카 YES 비율(%)', color='green', linestyle='--', marker='^')

    # ✅ 비율 % 라벨 표시
    for i, value in enumerate(daily_df['내배카 YES 비율(%)']):
        ax2.text(daily_df.index[i], value + 1, f"{value:.1f}%", ha='center', fontsize=8, color='green')

    ax2.set_ylabel("일일 내배카 YES 비율 (%)")
    ax2.legend(loc='upper right')

    # ▶ 제목 및 레이아웃
    plt.title(f"{file_name} - 누적 지원 현황 + 내배카 비율 추이")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import numpy as np
from datetime import datetime

# 🔠 한글 폰트 설정
if platform.system() == 'Windows':
    font_path = "C:/Windows/Fonts/malgun.ttf"
elif platform.system() == 'Darwin':
    font_path = "/System/Library/Fonts/AppleGothic.ttf"
else:
    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 폰트 파일이 없을 경우 오류가 발생할 수 있습니다. 해당 경로에 폰트가 있는지 확인해주세요.
try:
    fontprop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = fontprop.get_name()
except FileNotFoundError:
    print(f"'{font_path}' 경로에 폰트 파일이 없어 기본 폰트로 설정됩니다. 한글이 깨질 수 있습니다.")
plt.rcParams['axes.unicode_minus'] = False

# 📂 폴더 경로
folder_path = r"C:\Users\manid\OneDrive\바탕 화면\data_study\kdt"
# 폴더가 존재하지 않을 경우 오류 방지를 위해 확인 로직을 추가할 수 있습니다.
if not os.path.isdir(folder_path):
    print(f"오류: '{folder_path}' 폴더를 찾을 수 없습니다. 스크립트를 종료합니다.")
    exit() # 폴더가 없으면 스크립트 종료

csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

plt.style.use('ggplot')

# 🔁 파일 반복
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)

    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp949')
    df = df.fillna('')

    # 날짜 변환
    df['최초작성일'] = pd.to_datetime(df['최초작성일'], errors='coerce')
    df['생년월일'] = pd.to_datetime(df['생년월일'], errors='coerce')

    # 내부직원 필터링
    is_test_reason = df['지원취소 사유'].str.lower().str.contains('test|테스트', case=False)
    is_internal_email = df['지원서 이메일'].str.contains('likelion.net', case=False) | df['가입 이메일'].str.contains('likelion.net', case=False)
    is_internal = is_test_reason | is_internal_email

    # 조건 정의: 지원중
    cond_applying = df['합불상태'].isin(['검토전', '대상아님']) & (~is_internal)
    df_applying = df[cond_applying].copy()

    # 만나이 계산 함수
    def calculate_age(birthdate, ref_date):
        if pd.isnull(birthdate) or pd.isnull(ref_date):
            return np.nan
        return ref_date.year - birthdate.year - ((ref_date.month, ref_date.day) < (birthdate.month, birthdate.day))

    df_applying['만나이'] = df_applying.apply(lambda x: calculate_age(x['생년월일'], x['최초작성일']), axis=1)
    df_applying = df_applying.dropna(subset=['최초작성일', '만나이'])

    # 데이터가 없는 경우 다음 파일로 넘어감
    if df_applying.empty:
        print(f"'{file_name}'에서 처리할 데이터가 없습니다.")
        continue
        
    # ▶ 날짜별 나이 리스트 구성
    grouped = df_applying.groupby(df_applying['최초작성일'].dt.date)['만나이'].apply(list)

    # 그룹화된 데이터가 없는 경우 다음 파일로 넘어감
    if grouped.empty:
        print(f"'{file_name}'에서 그룹화할 데이터가 없습니다.")
        continue

    # ▶ 전체 중앙값 계산
    overall_median = df_applying['만나이'].median()

    # ▶ 박스플롯 시각화
    fig, ax = plt.subplots(figsize=(14, 6))
    box = ax.boxplot(grouped.tolist(), patch_artist=True, labels=grouped.index, showmeans=False)

    ax.axhline(overall_median, color='gray', linestyle='--', label=f'전체 중앙값: {overall_median:.1f}')
    ax.set_xticks(range(1, len(grouped.index) + 1))
    ax.set_xticklabels(grouped.index, rotation=45)
    ax.set_title(f'{file_name} - 지원중 유저 만나이 분포 (박스플롯)')
    ax.set_ylabel('만나이')

    # ▶ 각 날짜별 1Q, 3Q 레이블 추가
    for i, date in enumerate(grouped.index):
        q1 = np.percentile(grouped[date], 25)
        q3 = np.percentile(grouped[date], 75)
        ax.text(i+1, q1-1, f'Q1: {q1:.1f}', ha='center', fontsize=8, color='blue')
        ax.text(i+1, q3+1, f'Q3: {q3:.1f}', ha='center', fontsize=8, color='red')
        
        # ★★★ 추가된 부분 START ★★★
        # 1. 각 날짜(박스플롯)의 표본 개수 계산
        sample_count = len(grouped[date])
        # 2. 3사분위수(Q3)보다 약간 높은 위치에 표본 개수(n) 텍스트로 추가
        ax.text(i + 1, q3 + 2.5, f'n={sample_count}', ha='center', fontsize=9, color='black')
        # ★★★ 추가된 부분 END ★★★

    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import numpy as np

# 🔠 한글 폰트 설정
if platform.system() == 'Windows':
    font_path = "C:/Windows/Fonts/malgun.ttf"
elif platform.system() == 'Darwin':
    font_path = "/System/Library/Fonts/AppleGothic.ttf"
else:
    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

fontprop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = fontprop.get_name()
plt.rcParams['axes.unicode_minus'] = False

# 📂 폴더 경로
folder_path = r"C:\Users\manid\OneDrive\바탕 화면\data_study\kdt"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

plt.style.use('ggplot')

# 🔁 파일 반복
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)

    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp949')
    df = df.fillna('')

    # 날짜 변환
    df['최초작성일'] = pd.to_datetime(df['최초작성일'], errors='coerce')
    df['생년월일'] = pd.to_datetime(df['생년월일'], errors='coerce')

    # 내부직원 필터링
    is_test_reason = df['지원취소 사유'].str.lower().str.contains('test|테스트', case=False)
    is_internal_email = df['지원서 이메일'].str.contains('likelion.net', case=False) | df['가입 이메일'].str.contains('likelion.net', case=False)
    is_internal = is_test_reason | is_internal_email

    # 조건 정의: 지원중
    cond_applying = df['합불상태'].isin(['검토전', '대상아님']) & (~is_internal)
    df_applying = df[cond_applying].copy()

    # 만나이 계산 함수
    def calculate_age(birthdate, ref_date):
        if pd.isnull(birthdate) or pd.isnull(ref_date):
            return np.nan
        return ref_date.year - birthdate.year - ((ref_date.month, ref_date.day) < (birthdate.month, birthdate.day))

    df_applying['만나이'] = df_applying.apply(lambda x: calculate_age(x['생년월일'], x['최초작성일']), axis=1)
    df_applying = df_applying.dropna(subset=['최초작성일', '만나이', '성별'])
    df_applying = df_applying[df_applying['성별'].isin(['male', 'female'])]

    # ▶ 성별 분리
    grouped_male = df_applying[df_applying['성별'] == 'male'].groupby(df_applying['최초작성일'].dt.date)['만나이'].apply(list)
    grouped_female = df_applying[df_applying['성별'] == 'female'].groupby(df_applying['최초작성일'].dt.date)['만나이'].apply(list)

    median_male = df_applying[df_applying['성별'] == 'male']['만나이'].median()
    median_female = df_applying[df_applying['성별'] == 'female']['만나이'].median()

    # ▶ 시각화
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 10), sharex=True)

    for ax, grouped, gender, median in zip(
        axes,
        [grouped_male, grouped_female],
        ['남성', '여성'],
        [median_male, median_female]
    ):
        box = ax.boxplot(grouped.tolist(), patch_artist=True, labels=grouped.index, showmeans=False)

        ax.axhline(median, color='gray', linestyle='--', label=f'{gender} 전체 중앙값: {median:.1f}')
        ax.set_title(f"{file_name} - 지원중 유저 만나이 분포 ({gender})")
        ax.set_ylabel('만나이')
        ax.set_xticks(range(1, len(grouped.index) + 1))
        ax.set_xticklabels(grouped.index, rotation=45)

        # ▶ 1Q / 3Q 텍스트 라벨 추가
        for i, date in enumerate(grouped.index):
            q1 = np.percentile(grouped[date], 25)
            q3 = np.percentile(grouped[date], 75)
            ax.text(i+1, q1-1, f'Q1: {q1:.1f}', ha='center', fontsize=8, color='blue')
            ax.text(i+1, q3+1, f'Q3: {q3:.1f}', ha='center', fontsize=8, color='red')

        ax.legend()

    plt.suptitle(f"{file_name} - 성별 지원중 유저 만나이 분석", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


# 

In [ ]:
# -*- coding: utf-8 -*-
# 각 CSV(=부트캠프) 개별 예측. 부트캠프별 end_date/total_budget/spent_to_date 를 코드 상단에서 직접 입력.
import os, platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ========= ✅ 사용자 입력 =========
FOLDER_PATH = r"C:\Users\manid\OneDrive\바탕 화면\data_study\kdt"

# 예측 모델 선택: 'saturation' | 'dow_elasticity' | 'bayes_cr'
MODEL = 'bayes_cr'

BOOTCAMP_CONFIG = {
    # 'CSV파일명(확장자 제외)': {개별설정}
    'kdt-ugm-6th_지원서_2025_10_22_13_35_30':  {'end_date': '2025-10-30', 'total_budget': 60_500_000, 'spent_to_date': 60_984_178},
    'kdt-design-7th_지원서_2025_10_22_13_35_59':   {'end_date': '2025-10-26', 'total_budget': 55_000_000, 'spent_to_date': 49_779_602},
    'kdt-python-2nd_지원서_2025_10_22_13_35_09':    {'end_date': '2025-11-02', 'total_budget': 55_000_000, 'spent_to_date': 29_832_098},
    'kdt-frontend-16th_지원서_2025_10_22_13_35_01':   {'end_date': '2025-11-18', 'total_budget': 65_000_000, 'spent_to_date': 38_899_500},
    'kdt-dataanalysis-7th_지원서_2025_10_22_13_35_17':   {'end_date': '2025-11-23', 'total_budget': 55_000_000, 'spent_to_date': 17_356_642},
    # 없는 파일명은 스킵됨
}

# 시나리오(보수성/낙관도 조절용) — 보수성 완화 디폴트
SCENARIOS = {
    '보수적(비관)': {'alpha_cac': 0.5, 'cr_uplift': -0.01},
    '기준(베이스)': {'alpha_cac': 0.3, 'cr_uplift':  0.00},
    '공격적(낙관)': {'alpha_cac': 0.2, 'cr_uplift':  0.01},
}
# =================================

OUT_DIR = os.path.join(FOLDER_PATH, "out"); os.makedirs(OUT_DIR, exist_ok=True)

# 한글 폰트
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# --- 예측모듈 로드(없으면 강화된 간이버전 사용: 3가지 모델 포함) ---
def _import_or_fallback():
    try:
        from forecast_kdt import forecast as _f
        return _f
    except Exception:
        # ===== 강화된 내부 fallback (saturation / dow_elasticity / bayes_cr) =====
        import numpy as np, pandas as pd
        from dataclasses import dataclass
        from typing import Dict

        @dataclass
        class ScenarioParams:
            name: str
            alpha_cac: float = 0.3
            cr_uplift: float = 0.0
            organic_floor: float = 0.2
            trend_weight: float = 0.6

        def _rolling_avg(s: pd.Series, w=7) -> float:
            return float(s.tail(w).mean()) if len(s) else 0.0

        def _safe_div(n, d, default=0.0):
            return default if (d == 0 or pd.isna(d)) else (n / d)

        def _ensure_daily(df):
            df = df.copy()
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date').set_index('date')
            if len(df)==0:
                # 빈 df 보호
                return pd.DataFrame(index=pd.DatetimeIndex([]), data={'spend':[], 'apply_start':[], 'apply_complete':[]})
            all_days = pd.date_range(df.index.min(), df.index.max(), freq='D')
            return df.reindex(all_days).fillna(0.0)

        # --- Saturation: starts = c + a*log(1 + b*spend_eff) ---
        def _fit_saturation(df: pd.DataFrame, window=28, min_points=10):
            sub = df.tail(window)
            x = sub['spend'].values.astype(float)
            y = sub['apply_start'].values.astype(float)
            m = (~np.isnan(x)) & (~np.isnan(y))
            x, y = x[m], y[m]
            if len(x) < min_points or np.all(x <= 0):
                return (0.0, 0.0, _rolling_avg(df['apply_start'], 7))
            c0 = max(0.0, np.percentile(y, 10))
            y_adj = np.clip(y - c0, 1e-6, None)
            b_grid = np.geomspace(1e-8, 1e-3, 30)
            best = None
            for b in b_grid:
                X = np.log1p(b * x)
                a = max(0.0, np.sum(X * y_adj) / (np.sum(X * X) + 1e-9))
                pred = a * X + c0
                mse = np.mean((pred - y)**2)
                if (best is None) or (mse < best[0]):
                    best = (mse, a, b, c0)
            _, a, b, c = best
            return (a, b, c)

        def _weekday_weights(df: pd.DataFrame, weeks=8):
            sub = df.tail(7*weeks)
            if len(sub) < 7:
                return np.ones(7)
            s = sub['apply_start'].copy()
            s.index = pd.DatetimeIndex(s.index)
            by_wd = s.groupby(s.index.weekday).mean().reindex(range(7)).fillna(s.mean())
            w = by_wd.values
            return w / (w.mean() + 1e-9)

        def forecast(df: pd.DataFrame, end_date: str, remaining_budget: float,
                     scenarios: Dict[str, Dict], model: str = 'saturation') -> pd.DataFrame:
            df = _ensure_daily(df)
            end_dt = pd.to_datetime(end_date)
            last_dt = df.index.max() if len(df.index)>0 else pd.Timestamp(end_dt)
            horizon = max((end_dt - last_dt).days, 0)

            # 최근 지표
            recent_cac = _safe_div(df['spend'].tail(7).sum(), df['apply_start'].tail(7).sum(), default=np.inf)
            recent_cr = _safe_div(df['apply_complete'].tail(7).sum(), df['apply_start'].tail(7).sum(), default=0.0)
            recent_spend_avg = _rolling_avg(df['spend'], 7)
            recent_start_avg = _rolling_avg(df['apply_start'], 7)

            rows=[]
            for name, overrides in scenarios.items():
                p = ScenarioParams(name=name, **overrides)
                days = max(horizon, 1)
                base_daily = remaining_budget / days if days > 0 else 0.0

                # 마진 CAC (완화)
                if recent_spend_avg <= 0:
                    degr = 1.0 + 0.7*p.alpha_cac
                else:
                    ratio = base_daily / max(recent_spend_avg, 1e-6)
                    # 과도 보수성 완화: 계수 0.7
                    degr = 1.0 + 0.7*p.alpha_cac * max(ratio - 1.0, 0.0)

                # 모델별 일일 starts 벡터
                if model == 'saturation':
                    a,b,c = _fit_saturation(df, window=28, min_points=10)
                    eff_daily_spend = base_daily / max(degr,1e-6)
                    organic_floor = p.organic_floor * max(recent_start_avg, 0.0)
                    starts_daily = c + a*np.log1p(b*eff_daily_spend)
                    starts_daily = max(starts_daily, organic_floor)
                    starts_vec = np.full(days, starts_daily, dtype=float)

                elif model == 'dow_elasticity':
                    wd_w = _weekday_weights(df, weeks=8)
                    # 요일 가중 분배
                    if len(df.index)>0:
                        start_wd = (int(df.index.max().weekday()) + 1) % 7
                    else:
                        start_wd = 0
                    w = np.array([wd_w[(start_wd + i) % 7] for i in range(days)], dtype=float)
                    w = w / (w.sum() + 1e-9)
                    plan_spend_vec = base_daily * days * w
                    beta = 0.8  # 지출 탄력도(합리적 범위)
                    k = _safe_div(recent_start_avg, (max(recent_spend_avg,1e-6)**beta), default=0.0)
                    organic_floor = p.organic_floor * max(recent_start_avg, 0.0)
                    starts_vec = organic_floor + k * (plan_spend_vec / max(degr,1e-6))**beta

                elif model == 'bayes_cr':
                    # 시작은 saturation과 유사(간단형), CR만 베이지안 평균으로 안정화
                    eff_daily_spend = base_daily / max(degr,1e-6)
                    organic_floor = p.organic_floor * max(recent_start_avg, 0.0)
                    # 최근 CAC 없으면 paid 0
                    paid = 0.0 if (np.isinf(recent_cac) or recent_cac<=0) else (eff_daily_spend / recent_cac)
                    starts_daily = max(organic_floor + paid, organic_floor)
                    starts_vec = np.full(days, starts_daily, dtype=float)
                    # 베이지안 CR (간단)
                    prior_alpha, prior_beta = 20.0, 40.0
                    s = float(df['apply_start'].tail(14).sum()); c = float(df['apply_complete'].tail(14).sum())
                    alpha = prior_alpha + c; beta = prior_beta + max(s-c,0.0)
                    recent_cr = alpha/(alpha+beta)  # 평균
                else:
                    raise ValueError("unknown model")

                cr = float(np.clip(recent_cr + p.cr_uplift, 0.0, 0.99))
                comps_vec = starts_vec * cr

                rows.append({
                    '시나리오': name,
                    '모델': model,
                    '예상_일수': int(days),
                    '계획_일일집행액(원)': int(round(base_daily)),
                    '마진_CAC_배수': round(float(degr), 3),
                    '추정_최종_CR(완료/시작)': round(float(cr), 3),
                    '모객마감_누적_지원시작(명)': int(round(df['apply_start'].sum() + starts_vec.sum())),
                    '모객마감_누적_지원완료(명)': int(round(df['apply_complete'].sum() + comps_vec.sum())),
                })
            return pd.DataFrame(rows)
        return forecast

forecast = _import_or_fallback()

def _load_one_csv(file_path:str) -> pd.DataFrame:
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp949')
    df = df.fillna('')
    df['최초작성일'] = pd.to_datetime(df['최초작성일'], errors='coerce')
    df['지원완료일'] = pd.to_datetime(df['지원완료일'], errors='coerce')
    # 내부/테스트 제외
    is_test = df['지원취소 사유'].str.lower().str.contains('test|테스트', na=False)
    is_internal = df['지원서 이메일'].str.contains('likelion.net', na=False) | df['가입 이메일'].str.contains('likelion.net', na=False)
    df = df[~(is_test | is_internal)].copy()
    # 조건
    cond_applying = df['합불상태'].isin(['검토전', '대상아님'])
    cond_completed_start = (df['지원상태']=='지원완료') & (df['지원완료일'].notna())
    cond_canceled_valid = (df['지원상태']=='지원취소') & (df['지원완료일'].notna())
    cond_completed = (df['지원완료일'].notna()) & (df['지원상태']!='지원취소')
    # 집계
    started = df[cond_applying | cond_completed_start | cond_canceled_valid] \
                .groupby(df['최초작성일'].dt.date).size().rename('apply_start')
    completed = df[cond_completed] \
                .groupby(df['지원완료일'].dt.date).size().rename('apply_complete')
    daily = pd.concat([started, completed], axis=1).fillna(0).astype(int).reset_index()
    daily.rename(columns={'index':'date'}, inplace=True)
    daily['date'] = pd.to_datetime(daily['date'])
    # 날짜 연속화 + spend=0 (일단 광고비 로그 없을 때 0으로)
    if len(daily) == 0:
        # 빈 데이터 보호
        daily = pd.DataFrame({'date': pd.to_datetime([]),
                              'apply_start': [], 'apply_complete': [], 'spend': []})
        return daily
    all_days = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
    daily = daily.set_index('date').reindex(all_days).fillna(0.0).rename_axis('date').reset_index()
    daily[['apply_start','apply_complete']] = daily[['apply_start','apply_complete']].astype(int)
    daily['spend'] = 0.0
    return daily

# 실행
csv_files = [f for f in os.listdir(FOLDER_PATH) if f.lower().endswith('.csv')]
summary_rows=[]
for fname in sorted(csv_files):
    bootcamp = os.path.splitext(fname)[0]
    if bootcamp not in BOOTCAMP_CONFIG:
        print(f"[SKIP] 설정 없음: {bootcamp}")
        continue

    cfg = BOOTCAMP_CONFIG[bootcamp]
    end_date = cfg['end_date']; total_budget = float(cfg['total_budget'])
    spent_to_date = float(cfg.get('spent_to_date', 0.0))
    remaining_budget = max(total_budget - spent_to_date, 0.0)

    fpath = os.path.join(FOLDER_PATH, fname)
    daily = _load_one_csv(fpath)

    proj = forecast(
        daily[['date','spend','apply_start','apply_complete']],
        end_date=end_date,
        remaining_budget=remaining_budget,
        scenarios=SCENARIOS,
        model=MODEL  # ⬅️ 한 줄로 모델 선택
    )
    proj.insert(0, '부트캠프명', bootcamp)
    proj.insert(2, '총예산(원)', int(total_budget))
    proj.insert(3, '누적집행액(원)', int(spent_to_date))
    proj.insert(4, '남은예산(원)', int(remaining_budget))

    out_csv = os.path.join(OUT_DIR, f"{bootcamp}_forecast_{MODEL}.csv")
    proj.to_csv(out_csv, index=False, encoding='utf-8-sig')

    # 그래프(과거 누적 + 베이스 라인)
    cum_s = daily['apply_start'].cumsum()
    cum_c = daily['apply_complete'].cumsum()
    plt.figure(figsize=(12,6))
    if len(daily) > 0:
        plt.plot(daily['date'], cum_s, label='누적 지원시작(과거)', color='tomato', marker='o')
        plt.plot(daily['date'], cum_c, label='누적 지원완료(과거)', color='steelblue', marker='s')
    
        # ✅ 현재 누적 레이블
        if len(cum_s) > 0:
            plt.text(daily['date'].iloc[-1], cum_s.iloc[-1] + 2, 
                     f"{int(cum_s.iloc[-1]):,}", color='tomato', ha='left', fontsize=9)
        if len(cum_c) > 0:
            plt.text(daily['date'].iloc[-1], cum_c.iloc[-1] + 2, 
                     f"{int(cum_c.iloc[-1]):,}", color='steelblue', ha='left', fontsize=9)
    
    if not proj.empty:
        base = proj[proj['시나리오'] == '기준(베이스)'].iloc[0]
        plt.axhline(base['모객마감_누적_지원시작(명)'], linestyle='--', color='tomato', label='예상 지원시작(마감)')
        plt.axhline(base['모객마감_누적_지원완료(명)'], linestyle='--', color='steelblue', label='예상 지원완료(마감)')
    
        # ✅ 예측 마감 레이블
        plt.text(daily['date'].iloc[-1], base['모객마감_누적_지원시작(명)'] + 2,
                 f"예상 {int(base['모객마감_누적_지원시작(명)']):,}", color='tomato', ha='right', fontsize=9, fontweight='bold')
        plt.text(daily['date'].iloc[-1], base['모객마감_누적_지원완료(명)'] + 2,
                 f"예상 {int(base['모객마감_누적_지원완료(명)']):,}", color='steelblue', ha='right', fontsize=9, fontweight='bold')
    
    ttl = f"[{bootcamp}] 추세+예측 | 모델:{MODEL} / 총예산:{int(total_budget):,} / 누적:{int(spent_to_date):,} / 남은:{int(remaining_budget):,} / 마감:{end_date}"
    plt.title(ttl)
    plt.xlabel("날짜")
    plt.ylabel("누적 인원")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{bootcamp}_trend_{MODEL}.png"), dpi=150)
    plt.close()


    summary_rows.append(proj)

if summary_rows:
    pd.concat(summary_rows, ignore_index=True).to_csv(
        os.path.join(OUT_DIR, f"_summary_all_bootcamps_{MODEL}.csv"),
        index=False, encoding='utf-8-sig'
    )
print(f"완료: out 폴더에 부트캠프별 예측 파일/그래프 저장 (MODEL={MODEL}).")
